# 02 — Clean QCEW Data

This notebook reads the raw QCEW annual ZIP files (2010–2024), extracts county-level records for the industries relevant to the minimum wage border-county analysis, and writes a combined panel to `data/intermediate/qcew_panel.parquet`.

**Industries kept (NAICS codes):**
| Code | Description |
|------|-------------|
| 722 | Food services and drinking places (**primary focus**) |
| 72 | Accommodation and food services (broader aggregate) |
| 44-45 | Retail trade |
| 721 | Accommodation |

**Output:** `data/intermediate/qcew_panel.parquet` (compressed, safe to commit to GitHub)

In [15]:
import zipfile
import io
import pandas as pd
from pathlib import Path

# Paths
ROOT = Path().resolve().parent
RAW_QCEW = ROOT / "data" / "raw" / "qcew"
INTERMEDIATE = ROOT / "data" / "intermediate"
INTERMEDIATE.mkdir(parents=True, exist_ok=True)
OUTPUT = INTERMEDIATE / "qcew_panel.parquet"

print("Root      :", ROOT)
print("Raw QCEW  :", RAW_QCEW)
print("Output    :", OUTPUT)
print("ZIP files :", len(list(RAW_QCEW.glob("*.zip"))))

Root      : /Users/ogechukwuezenwa/Documents/Duke_University_Projects/Spring_2026/IDS_701-Causal-Inference_Final
Raw QCEW  : /Users/ogechukwuezenwa/Documents/Duke_University_Projects/Spring_2026/IDS_701-Causal-Inference_Final/data/raw/qcew
Output    : /Users/ogechukwuezenwa/Documents/Duke_University_Projects/Spring_2026/IDS_701-Causal-Inference_Final/data/intermediate/qcew_panel.parquet
ZIP files : 15


## 1. Inspect one file to understand the structure

In [16]:
# Peek at one county CSV inside the 2023 ZIP
sample_zip = RAW_QCEW / "2023_annual_by_area.zip"
with zipfile.ZipFile(sample_zip) as zf:
    members = [m for m in zf.namelist() if m.endswith(".csv")]
    sample = pd.read_csv(io.StringIO(zf.read(members[1]).decode("utf-8")), dtype=str)

print("Shape:", sample.shape)
print("Columns:", sample.columns.tolist())
sample.head(3)

Shape: (965, 43)
Columns: ['area_fips', 'own_code', 'industry_code', 'agglvl_code', 'size_code', 'year', 'qtr', 'disclosure_code', 'area_title', 'own_title', 'industry_title', 'agglvl_title', 'size_title', 'annual_avg_estabs_count', 'annual_avg_emplvl', 'total_annual_wages', 'taxable_annual_wages', 'annual_contributions', 'annual_avg_wkly_wage', 'avg_annual_pay', 'lq_disclosure_code', 'lq_annual_avg_estabs_count', 'lq_annual_avg_emplvl', 'lq_total_annual_wages', 'lq_taxable_annual_wages', 'lq_annual_contributions', 'lq_annual_avg_wkly_wage', 'lq_avg_annual_pay', 'oty_disclosure_code', 'oty_annual_avg_estabs_count_chg', 'oty_annual_avg_estabs_count_pct_chg', 'oty_annual_avg_emplvl_chg', 'oty_annual_avg_emplvl_pct_chg', 'oty_total_annual_wages_chg', 'oty_total_annual_wages_pct_chg', 'oty_taxable_annual_wages_chg', 'oty_taxable_annual_wages_pct_chg', 'oty_annual_contributions_chg', 'oty_annual_contributions_pct_chg', 'oty_annual_avg_wkly_wage_chg', 'oty_annual_avg_wkly_wage_pct_chg', 'oty

,area_fips,own_code,industry_code,agglvl_code,size_code,year,qtr,disclosure_code,area_title,own_title,...,oty_total_annual_wages_chg,oty_total_annual_wages_pct_chg,oty_taxable_annual_wages_chg,oty_taxable_annual_wages_pct_chg,oty_annual_contributions_chg,oty_annual_contributions_pct_chg,oty_annual_avg_wkly_wage_chg,oty_annual_avg_wkly_wage_pct_chg,oty_avg_annual_pay_chg,oty_avg_annual_pay_pct_chg
0,01001,0,10,70,0,2023,A,NaN,"Autauga County, Alabama",Total Covered,...,54267839,10.1,1933187,2.2,-522704,-44.9,62,6.9,3254,7.0
1,01001,1,10,71,0,2023,A,NaN,"Autauga County, Alabama",Federal Government,...,811973,14.6,0,0.0,0,0.0,18,1.5,962,1.5
2,01001,1,102,72,0,2023,A,NaN,"Autauga County, Alabama",Federal Government,...,811973,14.6,0,0.0,0,0.0,18,1.5,962,1.5


In [17]:
# What industry codes are available? (look at a statewide file for a full list)
with zipfile.ZipFile(sample_zip) as zf:
    state_file = [m for m in zf.namelist() if "Statewide" in m][0]
    state_df = pd.read_csv(io.StringIO(zf.read(state_file).decode("utf-8")), dtype=str)

state_df[["industry_code", "industry_title"]].drop_duplicates().sort_values(
    "industry_code"
).head(40)

,industry_code,industry_title
0,10,"10 Total, all industries"
437,101,101 Goods-producing
810,1011,1011 Natural resources and mining
438,1012,1012 Construction
439,1013,1013 Manufacturing
2,102,102 Service-providing
3,1021,"1021 Trade, transportation, and utilities"
4,1022,1022 Information
5,1023,1023 Financial activities
6,1024,1024 Professional and business services


## 2. Define filtering logic

### Industry selection — justification

All four industries are **low-wage sectors** heavily reliant on minimum wage workers, making them the most sensitive to minimum wage policy changes. The selection is grounded directly in the project proposal and the broader minimum wage literature:

| NAICS Code | Industry | Role | Justification |
|------------|----------|------|---------------|
| 722 | Food services and drinking places | **Primary outcome** | *"We expect restaurant and food service industries to be the primary analytic focus because they are heavily exposed to minimum wage policy and are commonly studied in prior work."* This is the same industry used by Dube, Lester & Reich (2010) in the original border-county paper. |
| 72 | Accommodation and food services | Broader aggregate | The aggregate category that includes NAICS 722. Used for robustness checks to verify results hold at a higher level of aggregation. |
| 44-45 | Retail trade | Secondary industry | Explicitly mentioned in the proposal: *"sectors such as restaurants, retail, and hospitality."* Retail is heavily minimum-wage-dependent and allows us to test whether effects generalise beyond food service. |
| 721 | Accommodation | Secondary industry | Explicitly mentioned in the proposal: *"retail trade, accommodation, or broader leisure and hospitality categories."* Another low-wage sector with high exposure to minimum wage policy. |

The analysis will lead with **NAICS 722** and use the others as robustness and heterogeneity checks.

In [18]:
FIRST_YEAR = 2010
LAST_YEAR = 2024

# NAICS-based industry codes used in QCEW
TARGET_INDUSTRIES = {
    "722": "Food services and drinking places",  # PRIMARY focus
    "72": "Accommodation and food services",  # broader aggregate (includes 722)
    "44-45": "Retail trade",  # secondary
    "721": "Accommodation",  # secondary
}

# own_code 5 = Private sector
# For specific NAICS codes (722, 72, 44-45, 721), QCEW only stores private-sector
# rows at the county level — own_code 0 (Total Covered) only appears on top-level
# all-industry aggregations, not on individual NAICS codes.
# Food services and retail are overwhelmingly private, so this is exactly what we want.
OWN_CODE = "5"

# Columns to keep
KEEP_COLS = [
    "area_fips",
    "own_code",
    "industry_code",
    "year",
    "disclosure_code",
    "area_title",
    "own_title",
    "industry_title",
    "annual_avg_estabs_count",
    "annual_avg_emplvl",
    "total_annual_wages",
    "annual_avg_wkly_wage",
    "avg_annual_pay",
]


def is_county_fips(fips: str) -> bool:
    """True if FIPS is a county (5 digits, last 3 not 000)."""
    return len(fips) == 5 and fips[2:] != "000"


print("Target industries:")
for code, title in TARGET_INDUSTRIES.items():
    print(f"  {code:6s} — {title}")
print(f"\nOwnership filter: own_code = {OWN_CODE!r} (Private sector)")

Target industries:
  722    — Food services and drinking places
  72     — Accommodation and food services
  44-45  — Retail trade
  721    — Accommodation

Ownership filter: own_code = '5' (Private sector)


## 3. Process one year (sanity check before running all years)

In [19]:
def process_year(year: int) -> pd.DataFrame:
    zip_path = RAW_QCEW / f"{year}_annual_by_area.zip"
    if not zip_path.exists():
        print(f"  [missing] {zip_path.name}")
        return pd.DataFrame()

    frames = []
    with zipfile.ZipFile(zip_path) as zf:
        members = [m for m in zf.namelist() if m.endswith(".csv")]
        for member in members:
            try:
                df = pd.read_csv(
                    io.StringIO(zf.read(member).decode("utf-8")), dtype=str
                )
            except Exception:
                continue
            df = df[df["area_fips"].apply(is_county_fips)]
            df = df[
                df["industry_code"].isin(TARGET_INDUSTRIES.keys())
                & (df["own_code"] == OWN_CODE)
            ]
            if not df.empty:
                cols = [c for c in KEEP_COLS if c in df.columns]
                frames.append(df[cols])

    if not frames:
        return pd.DataFrame()

    result = pd.concat(frames, ignore_index=True)
    print(f'  {year}: {len(result):,} rows, {result["area_fips"].nunique():,} counties')
    return result


# Test on 2023 first
test = process_year(2023)
test.head(10)

  2023: 14,373 rows, 3,660 counties


,area_fips,own_code,industry_code,year,disclosure_code,area_title,own_title,industry_title,annual_avg_estabs_count,annual_avg_emplvl,total_annual_wages,annual_avg_wkly_wage,avg_annual_pay
0,01001,5,44-45,2023,NaN,"Autauga County, Alabama",Private,NAICS 44-45 Retail trade,151,1666,54767547,632,32880
1,01001,5,72,2023,NaN,"Autauga County, Alabama",Private,NAICS 72 Accommodation and food services,77,1299,25986933,385,20013
2,01001,5,721,2023,NaN,"Autauga County, Alabama",Private,NAICS 721 Accommodation,10,50,1229207,470,24462
3,01001,5,722,2023,NaN,"Autauga County, Alabama",Private,NAICS 722 Food services and drinking places,66,1248,24757726,381,19834
4,01003,5,44-45,2023,NaN,"Baldwin County, Alabama",Private,NAICS 44-45 Retail trade,1112,14925,565007876,728,37856
5,01003,5,72,2023,NaN,"Baldwin County, Alabama",Private,NAICS 72 Accommodation and food services,726,13772,387500067,541,28137
6,01003,5,721,2023,NaN,"Baldwin County, Alabama",Private,NAICS 721 Accommodation,98,2607,92837614,685,35611
7,01003,5,722,2023,NaN,"Baldwin County, Alabama",Private,NAICS 722 Food services and drinking places,629,11165,294662453,508,26392
8,01005,5,44-45,2023,NaN,"Barbour County, Alabama",Private,NAICS 44-45 Retail trade,84,890,26389821,570,29660
9,01005,5,72,2023,NaN,"Barbour County, Alabama",Private,NAICS 72 Accommodation and food services,53,767,13028690,327,16996


In [ ]:
# Diagnostic: check what industry codes actually appear in ONE county file
with zipfile.ZipFile(RAW_QCEW / "2023_annual_by_area.zip") as zf:
    # pick a large county (Cook County, IL = 17031) for a richer set of industries
    county_file = [m for m in zf.namelist() if "17031" in m]
    if not county_file:
        # fallback to first county file
        county_file = [m for m in zf.namelist() if m.endswith(".csv")][1]
    else:
        county_file = county_file[0]

    print("File:", county_file)
    df_county = pd.read_csv(
        io.StringIO(zf.read(county_file).decode("utf-8")), dtype=str
    )

print("Shape:", df_county.shape)
print("\nAll unique industry_codes in this county file:")
print(
    df_county[["industry_code", "industry_title"]]
    .drop_duplicates()
    .sort_values("industry_code")
    .to_string()
)
print("\nown_code values:", df_county["own_code"].unique())

File: 2023.annual.by_area/2023.annual 17031 Cook County, Illinois.csv
Shape: (2339, 43)

All unique industry_codes in this county file:
     industry_code                                                                                                                       industry_title
0               10                                                                                                             10 Total, all industries
2              101                                                                                                                  101 Goods-producing
380           1011                                                                                                    1011 Natural resources and mining
156           1012                                                                                                                    1012 Construction
3             1013                                                                                      

In [21]:
# Check value counts for industry codes in the test year
test[["industry_code", "industry_title"]].value_counts()

industry_code  industry_title                             
44-45          NAICS 44-45 Retail trade                       3657
72             NAICS 72 Accommodation and food services       3646
722            NAICS 722 Food services and drinking places    3639
721            NAICS 721 Accommodation                        3431
Name: count, dtype: int64

In [22]:
# Check how many disclosed vs suppressed cells (disclosure_code N = suppressed)
test["disclosure_code"].value_counts(dropna=False)

disclosure_code
NaN    11865
N       2508
Name: count, dtype: int64

## 4. Process all years and combine

In [23]:
print(f"=== Processing {FIRST_YEAR}–{LAST_YEAR} ===")
all_frames = []
for year in range(FIRST_YEAR, LAST_YEAR + 1):
    df = process_year(year)
    if not df.empty:
        all_frames.append(df)

print(f"\nDone. Total years processed: {len(all_frames)}")

=== Processing 2010–2024 ===
  2010: 14,297 rows, 3,638 counties
  2011: 14,293 rows, 3,638 counties
  2012: 14,294 rows, 3,638 counties
  2013: 14,370 rows, 3,658 counties
  2014: 14,376 rows, 3,657 counties
  2015: 14,358 rows, 3,659 counties
  2016: 14,355 rows, 3,659 counties
  2017: 14,349 rows, 3,658 counties
  2018: 14,356 rows, 3,659 counties
  2019: 14,355 rows, 3,659 counties
  2020: 14,356 rows, 3,661 counties
  2021: 14,363 rows, 3,661 counties
  2022: 14,373 rows, 3,660 counties
  2023: 14,373 rows, 3,660 counties
  2024: 14,383 rows, 3,665 counties

Done. Total years processed: 15


In [24]:
# Combine into one panel
panel = pd.concat(all_frames, ignore_index=True)

# Convert numeric columns
numeric_cols = [
    "annual_avg_estabs_count",
    "annual_avg_emplvl",
    "total_annual_wages",
    "annual_avg_wkly_wage",
    "avg_annual_pay",
]
for col in numeric_cols:
    if col in panel.columns:
        panel[col] = pd.to_numeric(panel[col], errors="coerce")
panel["year"] = pd.to_numeric(panel["year"], errors="coerce").astype("Int64")

# Sort
panel = panel.sort_values(["area_fips", "industry_code", "year"]).reset_index(drop=True)

print("Shape         :", panel.shape)
print("Counties      :", panel["area_fips"].nunique())
print("Years         :", sorted(panel["year"].dropna().unique().tolist()))
print("Industries    :")
print(
    panel[["industry_code", "industry_title"]].drop_duplicates().to_string(index=False)
)
panel.head()

Shape         : (215251, 13)
Counties      : 3711
Years         : [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
Industries    :
industry_code                              industry_title
        44-45                                Retail trade
        44-45                    NAICS 44-45 Retail trade
           72             Accommodation and food services
           72    NAICS 72 Accommodation and food services
          721                               Accommodation
          721                     NAICS 721 Accommodation
          722           Food services and drinking places
          722 NAICS 722 Food services and drinking places


,area_fips,own_code,industry_code,year,disclosure_code,area_title,own_title,industry_title,annual_avg_estabs_count,annual_avg_emplvl,total_annual_wages,annual_avg_wkly_wage,avg_annual_pay
0,01001,5,44-45,2010,NaN,"Autauga County, Alabama",Private,Retail trade,160,2041,45069777,425,22079
1,01001,5,44-45,2011,NaN,"Autauga County, Alabama",Private,Retail trade,156,2092,45424899,418,21713
2,01001,5,44-45,2012,NaN,"Autauga County, Alabama",Private,Retail trade,146,1738,39415219,436,22684
3,01001,5,44-45,2013,NaN,"Autauga County, Alabama",Private,Retail trade,145,1737,39519797,438,22751
4,01001,5,44-45,2014,NaN,"Autauga County, Alabama",Private,Retail trade,147,1776,40113646,434,22592


In [25]:
# Check for missing values
panel.isnull().sum()

area_fips                       0
own_code                        0
industry_code                   0
year                            0
disclosure_code            175638
area_title                      0
own_title                       0
industry_title                  0
annual_avg_estabs_count         0
annual_avg_emplvl               0
total_annual_wages              0
annual_avg_wkly_wage            0
avg_annual_pay                  0
dtype: int64

In [26]:
# Summary statistics
panel[numeric_cols].describe()

,annual_avg_estabs_count,annual_avg_emplvl,total_annual_wages,annual_avg_wkly_wage,avg_annual_pay
count,215251.000000,215251.000000,2.152510e+05,215251.000000,215251.000000
mean,314.852196,5047.621530,1.343522e+08,326.674631,16987.068534
std,1430.487609,22323.374757,7.231007e+08,221.530757,11519.642615
min,0.000000,0.000000,0.000000e+00,0.000000,0.000000
25%,15.000000,70.000000,1.078707e+06,224.000000,11649.000000
50%,47.000000,537.000000,9.787298e+06,322.000000,16738.000000
75%,172.000000,2525.000000,5.439000e+07,456.000000,23728.000000
max,75887.000000,944168.000000,4.158924e+10,10463.000000,544077.000000


## 5. Save to intermediate

In [27]:
# Save as Parquet — much smaller than CSV and safe to commit to GitHub
panel.to_parquet(OUTPUT, index=False)
print(f"Saved to  : {OUTPUT}")
print(f"File size : {OUTPUT.stat().st_size / 1e6:.1f} MB")
print(f"Rows      : {len(panel):,}")

# Also verify it reads back correctly
check = pd.read_parquet(OUTPUT)
print(f"Read back : {check.shape} — OK ✓")

Saved to  : /Users/ogechukwuezenwa/Documents/Duke_University_Projects/Spring_2026/IDS_701-Causal-Inference_Final/data/intermediate/qcew_panel.parquet
File size : 3.0 MB
Rows      : 215,251
Read back : (215251, 13) — OK ✓


## 6. Cleaning Notes & Findings

### Missing values
The only column with missing values is `disclosure_code` (175,638 NaN out of the full panel). This is **expected and not a problem** — in QCEW, a blank `disclosure_code` means the cell is **fully disclosed** (BLS did not suppress it). A value of `'N'` means the cell was suppressed for confidentiality. So the NaN rows are the *good* rows — they have real employment and wage data.

**Action in later notebooks:** when building the analysis panel, drop rows where `disclosure_code == 'N'` to avoid treating suppressed zeros as actual zeros.

### Panel coverage
- **4 industries** captured: NAICS 722 (food services, primary), 72 (accommodation & food, aggregate), 44-45 (retail), 721 (accommodation)
- **Ownership:** `own_code = 5` (Private sector) — correct for these industries since food service and retail are overwhelmingly private
- **Years:** 2010–2024 (15 years), giving enough pre/post variation across state minimum wage changes
- **No NaN** in any employment or wage column — the numeric conversion (`pd.to_numeric(..., errors='coerce')`) introduced no new missing values, confirming the raw data is clean for all disclosed cells
